# Lab 01 — Building a RAG pipeline with LangChain and LlamaIndex

**Level:** Beginner · **Duration:** ~75 minutes · **Covers:** H1 end-to-end RAG pipeline with LangChain (45 min) · H2 the same pipeline with LlamaIndex (30 min)

Lab guide: [`../lab-01-rag-pipeline-langchain-llamaindex.md`](../lab-01-rag-pipeline-langchain-llamaindex.md) — read each part there first; this notebook holds the code.

This lab is guided: every cell is given. You build a **naive RAG** pipeline twice over the same Daimler Truck after-sales corpus, first with LangChain and FAISS, then with LlamaIndex and Chroma, and record what you observe in the tables. Every document is synthetic.

In [ ]:
# (notebook) Setup: run this cell first. LLM_PROVIDER picks the model, EMBEDDING_PROVIDER the embeddings.
import json
import os
import sys
import time
import warnings
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # walks up from this folder to the repo-root .env
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")  # Chroma: no usage telemetry
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="IProgress not found")
pd.set_option("display.max_colwidth", 110)

# labs/module-03 holds data/ and rag_kit.py; the notebook runs from notebooks/ or solutions/.
LAB_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents) if p.name == "module-03" and (p / "data").is_dir())
DATA_DIR, OUTPUT_DIR = LAB_DIR / "data", LAB_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
sys.path[:0] = [str(LAB_DIR.parent / "setup"), str(LAB_DIR)]
import rag_kit as rk  # noqa: E402
from llm import (  # noqa: E402
    describe, embedding_slug, get_embedding_model_name, get_embedding_provider, get_provider,
    is_configured, partner_provider,
)
from transformers.utils import logging as hf_logging  # noqa: E402

hf_logging.disable_progress_bar()
PROVIDER, PARTNER = get_provider(), partner_provider()
if not is_configured(PROVIDER):
    raise SystemExit(
        f"❌ LLM_PROVIDER={PROVIDER} is not configured. Add its key and model to the repo-root .env "
        "(template: labs/setup/.env.example), restart the kernel and run this cell again."
    )
HAS_PARTNER = is_configured(PARTNER)


def check(name: str, fn) -> None:
    """Run a check function. A returned string means 'skipped'; a TODO is reported, not raised."""
    try:
        skipped = fn()
    except NotImplementedError as exc:
        print(f"⏳ {name}: TODO not implemented yet ({exc})")
        return
    print(f"⚠️  {name}: skipped ({skipped})" if skipped else f"✅ {name}")


print(describe())
print(f"Comparison provider: {PARTNER} {'✅' if HAS_PARTNER else '⚠️  not configured: comparison cells skip it'}")
print(f"Embeddings: {get_embedding_provider()} · {get_embedding_model_name()} (index suffix {embedding_slug()})")

import faiss
import numpy as np
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from llm import get_chat_model, get_embeddings


def chat_model(provider: str):
    """LangChain chat model; low reasoning effort on OpenAI's own API."""
    return get_chat_model(provider, **({"reasoning": {"effort": "low"}} if provider == "openai" else {}))


llm = chat_model(PROVIDER)
QUESTION = "An LH-40 built in March 2026 shows FC-ADB-0412. Which bulletin applies, and which part replaces the dosing valve?"

## Part 1 — The corpus, and a question without retrieval (10 min)

Twenty synthetic Daimler Truck documents: technical service bulletins (`TSB-…`), workshop repair
guides and customer FAQs (`KB-…`), and a slice of dealer and plant IT runbooks. Two are in German.

In [ ]:
# (notebook) One record per Markdown file; the front matter becomes metadata.
records = rk.load_corpus()
pd.DataFrame([{k: r["metadata"][k] for k in ("doc_id", "doc_type", "product", "audience", "language", "title")} for r in records])

In [ ]:
# (notebook) The model alone: it has never seen these internal documents.
print(llm.invoke(QUESTION).text)

## Part 2 — Naive RAG with LangChain: chunk, embed and index in FAISS (15 min)

**Ingest → chunk → embed → index.** FAISS is a vector-search *library*: it stores vectors only,
so you keep the chunks and their metadata yourself.

In [ ]:
# (notebook) Chunk: LangChain's splitter counts CHARACTERS. add_start_index gives every chunk a stable offset.
docs = [Document(page_content=r["text"], metadata=r["metadata"]) for r in records]
splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80, add_start_index=True)
chunks = splitter.split_documents(docs)
sizes = [len(c.page_content) for c in chunks]
print(f"{len(docs)} documents -> {len(chunks)} chunks · median {int(np.median(sizes))} characters · max {max(sizes)}")

In [ ]:
# (notebook) Embed and index. Unit-length vectors, so inner product = cosine similarity.
embeddings = get_embeddings()  # EMBEDDING_PROVIDER: local Sentence Transformers by default
vectors = np.array(embeddings.embed_documents([c.page_content for c in chunks]), dtype="float32")
faiss_index = faiss.IndexFlatIP(vectors.shape[1])  # exact search; fine up to ~1M vectors
faiss_index.add(vectors)

rk.FAISS_DIR.mkdir(exist_ok=True)  # git-ignored
faiss.write_index(faiss_index, str(rk.FAISS_DIR / "lab01.faiss"))
(rk.FAISS_DIR / "lab01-chunks.json").write_text(json.dumps([{"text": c.page_content, "metadata": c.metadata} for c in chunks]))
print(f"FAISS: {faiss_index.ntotal} vectors of dimension {faiss_index.d} · saved to {rk.FAISS_DIR.name}/")


def retrieve(question: str, k: int = 4) -> list[Document]:
    """Top-k chunks for a question: embed it, search FAISS, map row numbers back to chunks."""
    query = np.array([embeddings.embed_query(question)], dtype="float32")
    scores, rows = faiss_index.search(query, k)
    return [Document(page_content=chunks[i].page_content, metadata={**chunks[i].metadata, "score": float(s)})
            for s, i in zip(scores[0], rows[0])]


for hit in retrieve(QUESTION):
    print(f"{hit.metadata['score']:.2f}  {hit.metadata['doc_id']:13} {hit.page_content[:70]!r}")

In [ ]:
# (notebook) Check: every chunk is indexed, and the right documents come back for the question.
def check_part2():
    assert faiss_index.ntotal == len(chunks)
    top = [h.metadata["doc_id"] for h in retrieve(QUESTION)]
    assert {"TSB-2026-031", "KB-0610"} & set(top), top


check("Part 2 — chunked, embedded and indexed in FAISS", check_part2)

## Part 3 — Retrieve, augment, generate, with citations (20 min)

An LCEL chain: the retriever fills `{sources}`, the prompt asks for a citation after every claim,
and your code checks that every cited ID was actually retrieved.

In [ ]:
# (notebook) Augment: one <source> per document, its chunks in document order, so no id appears twice.
def format_sources(hits: list[Document]) -> str:
    by_doc: dict[str, list[Document]] = {}
    for h in hits:
        by_doc.setdefault(h.metadata["doc_id"], []).append(h)
    return "\n\n".join(
        f'<source id="{doc_id}" title="{parts[0].metadata["title"]}">\n'
        + "\n[...]\n".join(p.page_content for p in sorted(parts, key=lambda p: p.metadata["start_index"]))
        + "\n</source>"
        for doc_id, parts in by_doc.items()
    )


RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You are the Daimler Truck after-sales assistant. Answer only from the sources and cite the "
               "source id in square brackets after each claim, for example [KB-0610]. If the sources do not "
               "answer the question, say so."),
    ("human", "<sources>\n{sources}\n</sources>\n\nQuestion: {question}"),
])


def rag_chain(model):
    return {"sources": RunnableLambda(retrieve) | format_sources, "question": RunnablePassthrough()} | RAG_PROMPT | model | StrOutputParser()


answer = rag_chain(llm).invoke(QUESTION)
retrieved = {h.metadata["doc_id"] for h in retrieve(QUESTION)}
print(answer, "\n")
print(f"cited: {sorted(rk.cited_ids(answer))} · retrieved: {sorted(retrieved)}")

In [ ]:
# (notebook) Three more questions: a paraphrase, a German question, and one the corpus cannot answer.
QUESTIONS = {
    "paraphrase": "The exhaust fluid light is flashing. Can the driver keep going?",
    "German": "Was bedeutet der Fehlercode FC-BRK-0705?",
    "unanswerable": "What is the list price of a new LH-40 tractor?",
}
runs = []
for kind, question in QUESTIONS.items():
    text = rag_chain(llm).invoke(question)
    runs.append({"kind": kind, "retrieved": sorted({h.metadata["doc_id"] for h in retrieve(question)}),
                 "cited": sorted(rk.cited_ids(text)), "answer": text[:150]})
pd.DataFrame(runs)

In [ ]:
# (notebook) The same chain on the other provider: only the model object changes.
if not HAS_PARTNER:
    print(f"⚠️  {PARTNER} not configured: skipping the comparison")
else:
    other = rag_chain(chat_model(PARTNER)).invoke(QUESTION)
    print(f"{PARTNER}:\n{other}\n\ncited: {sorted(rk.cited_ids(other))}")

In [ ]:
# (notebook) Check: the answer cites sources, and every cited id was actually retrieved.
def check_part3():
    assert rk.cited_ids(answer), "the answer should cite at least one source"
    assert rk.cited_ids(answer) <= retrieved, f"cited ids that were not retrieved: {rk.cited_ids(answer) - retrieved}"
    for run in runs:
        assert set(run["cited"]) <= set(run["retrieved"]), run


check("Part 3 — grounded answers with valid citations", check_part3)

## Part 4 — The same pipeline in LlamaIndex, on Chroma (20 min)

LlamaIndex is built around the data side: **Documents → nodes → index → query engine**. Chroma is a
vector *database*: it persists the vectors **and** the metadata, so later labs can filter on it.

In [ ]:
# (notebook) Same records, same embedding model. SentenceSplitter counts TOKENS, not characters.
from llama_index.core import PromptTemplate, StorageContext, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter

rk.setup_llama(PROVIDER)  # Settings.llm and Settings.embed_model from .env
li_docs = rk.to_llama_documents(records)  # id_ = doc_id; only the title is embedded with each chunk
nodes = SentenceSplitter(chunk_size=160, chunk_overlap=20).get_nodes_from_documents(li_docs)
store = rk.chroma_vector_store(f"lab01-{embedding_slug()}")  # chroma_db/ (git-ignored)
index = VectorStoreIndex(nodes, storage_context=StorageContext.from_defaults(vector_store=store))
print(f"{len(li_docs)} documents -> {len(nodes)} nodes · Chroma collection holds {store._collection.count()} vectors")

In [ ]:
# (notebook) A query engine = retriever + prompt + LLM. The default prompt does not ask for citations, so pass one.
QA_PROMPT = PromptTemplate(
    "You are the Daimler Truck after-sales assistant. Answer only from the sources and cite the doc_id in "
    "square brackets after each claim, for example [KB-0610]. If the sources do not answer, say so.\n\n"
    "<sources>\n{context_str}\n</sources>\n\nQuestion: {query_str}\nAnswer: "
)
query_engine = index.as_query_engine(similarity_top_k=4, text_qa_template=QA_PROMPT)
response = query_engine.query(QUESTION)
li_retrieved = {n.node.metadata["doc_id"] for n in response.source_nodes}
print(response, "\n")
print("sources:", [(n.node.metadata["doc_id"], round(n.score, 2)) for n in response.source_nodes])

In [ ]:
# (notebook) Check: Chroma holds every node, and the LlamaIndex answer cites only what it retrieved.
def check_part4():
    assert store._collection.count() == len(nodes)
    cited = rk.cited_ids(str(response))
    assert cited and cited <= li_retrieved, (cited, li_retrieved)


check("Part 4 — LlamaIndex pipeline on Chroma", check_part4)

## Part 5 — Compare the two pipelines (10 min)

In [ ]:
# (notebook) Same questions, both pipelines: which documents did each retrieve?
li_retriever = index.as_retriever(similarity_top_k=4)
rows = []
for question in [QUESTION, *QUESTIONS.values()]:
    lc_ids = [h.metadata["doc_id"] for h in retrieve(question)]
    li_ids = rk.doc_ids(li_retriever.retrieve(question))
    rows.append({"question": question[:55], "LangChain + FAISS": lc_ids, "LlamaIndex + Chroma": li_ids,
                 "shared docs": len(set(lc_ids) & set(li_ids))})
pd.DataFrame(rows)

📝 **Observation table.** Fill it in from your run.

| | LangChain + FAISS | LlamaIndex + Chroma |
| --- | --- | --- |
| Lines of pipeline code you wrote | | |
| Chunk size unit and count | characters · | tokens · |
| Where metadata lives | | |
| Citations valid on both providers? | | |
| Which would you pick for this corpus, and why? | | |

---

**Done.** Answer the checkpoint questions in the lab guide, then go on to
[Lab 02 — Tuning and evaluating retrieval](../lab-02-tuning-evaluating-retrieval.md).